# Maternal Health Monitoring — LSTM (Google Colab)

Run the research LSTM model directly from the [GitHub repository](https://github.com/kennygeemartin/Maternal_Health_Monitoring_LSTM).

**Instructions:** Choose **Runtime → Run all**. The first cell installs the project. The next cell attempts to fetch the publicly published Mendeley dataset; if that fails, you can upload the official `Mathernal_Risk.csv` file (from [Mendeley Data](https://data.mendeley.com/datasets/ckhgtgdg4f/1)) to your private Colab session. The final run cell trains and reports test metrics.

**Privacy:** Source CSV includes names and patient identifiers. Do not commit the CSV to the public GitHub repository or publish individual records/output. This notebook is for research and education, not clinical diagnosis or treatment decisions.

## 1. Install dependencies and obtain the latest project code

Rerunning this cell updates an already-cloned repository instead of failing because the folder exists. CPU runtime is sufficient; no GPU is required.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

repo_url = "https://github.com/kennygeemartin/Maternal_Health_Monitoring_LSTM.git"
repo_dir = Path("/content/Maternal_Health_Monitoring_LSTM")
if repo_dir.exists():
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", repo_url, str(repo_dir)], check=True)

os.chdir(repo_dir)
if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Project ready at:", repo_dir)

## 2. Obtain dataset and check data split

The code downloads and caches the source dataset as `maternal_health_mendeley.csv`. If public API access fails, a file picker appears; upload the original Mendeley CSV. **The file stays in your Colab runtime and is not pushed to GitHub.**

In [ ]:
import io
import pandas as pd
from maternal_risk_lstm import fetch_mendeley_dataset, prepare_data

try:
    df = fetch_mendeley_dataset()
except Exception as download_error:
    print("Automatic dataset download unavailable:", type(download_error).__name__)
    print("Please upload Mathernal_Risk.csv from https://data.mendeley.com/datasets/ckhgtgdg4f/1")
    from google.colab import files
    uploads = files.upload()
    filename = next((n for n in uploads if n.lower().endswith((".csv", ".xlsx", ".xls"))), None)
    if filename is None:
        raise RuntimeError("Upload a .csv, .xlsx or .xls dataset file to continue.") from download_error
    df = (pd.read_csv(io.BytesIO(uploads[filename])) if filename.lower().endswith(".csv")
          else pd.read_excel(io.BytesIO(uploads[filename])))
    df.to_csv("maternal_health_mendeley.csv", index=False)

(train_x, train_y), (val_x, val_y), (test_x, test_y), scaler = prepare_data(df)
print("Loaded records:", len(df))
print("Train / validation / test:", len(train_y), len(val_y), len(test_y))
print("Model input shape:", train_x.shape)
print("Only aggregate counts and shapes are displayed; individual patient records are not printed.")

## 3. Train the model and evaluate on the held-out test split

This runs the existing repository script unchanged, with 180 maximum epochs and early stopping after 15 non-improving validation epochs. The final cell saves the console report as `maternal_lstm_results.txt` for download. Results depend on the dataset and runtime; metrics in the README are reference values, not guaranteed outputs.

In [ ]:
import subprocess
import sys

report_file = "maternal_lstm_results.txt"
with open(report_file, "w", encoding="utf-8") as report:
    process = subprocess.Popen(
        [sys.executable, "-u", "maternal_risk_lstm.py"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    for line in process.stdout:
        print(line, end="")
        report.write(line)
    exit_code = process.wait()
if exit_code != 0:
    raise RuntimeError(f"Training exited with status {exit_code}. Inspect the output above.")
print("\nTraining output saved:", report_file)

## 4. Optional: download the aggregate training results

Only the printed evaluation report is downloaded; raw patient data remains in the Colab runtime.

In [ ]:
from google.colab import files
files.download("maternal_lstm_results.txt")